## PS3 - robot strategy battle

TA clarified the eval formula after the PDF went out:
`Evaluation = (MAX Score - MIN Score) + Positional Advantage`, where positional advantage is manhattan(B, nearest E) - manhattan(A, nearest E), 0 if no energy left.

In [1]:
import time

MOVES_NORMAL = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]

def find_positions(grid, r, c):
    a_pos = b_pos = None
    energy = set()
    for i in range(r):
        for j in range(c):
            ch = grid[i][j]
            if ch == "A": a_pos = (i, j)
            elif ch == "B": b_pos = (i, j)
            elif ch == "E": energy.add((i, j))
    return a_pos, b_pos, energy

def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C

In [2]:
def manhattan(p1, p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

print(manhattan((0,0), (3,2)))
print(manhattan((1,1), (1,1)))

5
0


In [3]:
def nearest_dist(pos, energy):
    if not energy:
        return 0
    return min(manhattan(pos, e) for e in energy)

def evaluate(a_pos, b_pos, energy, max_score, min_score):
    adv = nearest_dist(b_pos, energy) - nearest_dist(a_pos, energy) if energy else 0
    return (max_score - min_score) + adv

In [4]:
def get_moves(pos, grid, R, C, energy, use_heuristic_order):
    moves = []
    for name, dr, dc in MOVES_NORMAL:
        nr, nc = pos[0] + dr, pos[1] + dc
        if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#":
            moves.append((name, nr, nc))
    if use_heuristic_order:
        moves.sort(key=lambda m: 0 if (m[1], m[2]) in energy else 1)
    return moves

### minimax / alpha-beta

In [5]:
def search(grid, R, C, a_pos, b_pos, energy, turn, depth, use_ab, use_heuristic_order):
    stats = {"generated": 0, "expanded": 0, "pruned": 0}

    def recurse(a_pos, b_pos, energy, max_score, min_score, turn, depth, alpha, beta):
        stats["expanded"] += 1
        cur_pos = a_pos if turn == "MAX" else b_pos
        moves = get_moves(cur_pos, grid, R, C, energy, use_heuristic_order)
        if depth == 0 or not moves:
            return evaluate(a_pos, b_pos, energy, max_score, min_score), None

        maximizing = turn == "MAX"
        best_val = float("-inf") if maximizing else float("inf")
        best_move = moves[0][0]

        for i, (name, nr, nc) in enumerate(moves):
            stats["generated"] += 1
            new_a, new_b, new_energy = a_pos, b_pos, energy
            new_max, new_min = max_score, min_score
            if turn == "MAX":
                new_a = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_max = max_score + 10
            else:
                new_b = (nr, nc)
                if (nr, nc) in energy:
                    new_energy = energy - {(nr, nc)}
                    new_min = min_score + 10

            next_turn = "MIN" if turn == "MAX" else "MAX"
            val, _ = recurse(new_a, new_b, new_energy, new_max, new_min, next_turn, depth - 1, alpha, beta)

            if maximizing and val > best_val:
                best_val, best_move = val, name
            if (not maximizing) and val < best_val:
                best_val, best_move = val, name

            if use_ab:
                if maximizing: alpha = max(alpha, best_val)
                else: beta = min(beta, best_val)
                if alpha >= beta:
                    stats["pruned"] += len(moves) - i - 1
                    break
        return best_val, best_move

    value, move = recurse(a_pos, b_pos, energy, 0, 0, turn, depth, float("-inf"), float("inf"))
    return value, move, stats

In [6]:
# hand check: A.E / ... / B.. , depth 2. A's only moves are RIGHT or DOWN.
# RIGHT -> either B reply gives eval 2. DOWN -> either B reply gives eval 0. so MAX picks RIGHT, value 2.
grid = ['A.E', '...', 'B..']
a_pos, b_pos, energy = find_positions(grid, 3, 3)
print(search(grid, 3, 3, a_pos, b_pos, energy, 'MAX', 2, False, False))

(2, 'RIGHT', {'generated': 6, 'expanded': 7, 'pruned': 0})


In [7]:
print(search(grid, 3, 3, a_pos, b_pos, energy, 'MAX', 2, True, False))  # alpha-beta should agree

(2, 'RIGHT', {'generated': 5, 'expanded': 6, 'pruned': 1})


### the real board

In [8]:
sample1 = '''5 7\nA.E.#..\n.#..#E.\n...#...\n#E#..#.\n....E.B\nMAX\n5'''.split(chr(10))
grid1 = sample1[1:6]
a1, b1, e1 = find_positions(grid1, 5, 7)
print(search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, False, False)[:2])
print(search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, True, False)[:2])
# Best Move matches the PDF's example (RIGHT). PDF shows Score = 20 but that's printed
# before the eval formula is even defined in the doc, real value here is 1

(1, 'RIGHT')
(1, 'RIGHT')


In [9]:
sample2 = '''5 7\nA...E..\n.###...\n..E....\n...###.\n.E...B.\nMAX\n4'''.split(chr(10))
grid2 = sample2[1:6]
a2, b2, e2 = find_positions(grid2, 5, 7)
print(search(grid2, 5, 7, a2, b2, e2, 'MAX', 4, False, False)[:2])
print(search(grid2, 5, 7, a2, b2, e2, 'MAX', 4, True, False)[:2])

(0, 'RIGHT')
(0, 'RIGHT')


### move ordering, does it actually matter

In [10]:
# both boards above give identical prune counts for heuristic vs normal order.
# turns out A's first move is already RIGHT on both (UP is invalid off the top edge),
# so the energy-first reorder never gets a chance to change anything near the root.
# built a board where that's not true: A's energy neighbor is DOWN, 3rd in normal order
grid3 = ['.....', '.A...', '.E...', '.....', '....B']
a3, b3, e3 = find_positions(grid3, 5, 5)
print('normal:   ', search(grid3, 5, 5, a3, b3, e3, 'MAX', 5, True, False))
print('heuristic:', search(grid3, 5, 5, a3, b3, e3, 'MAX', 5, True, True))

normal:    (10, 'UP', {'generated': 121, 'expanded': 122, 'pruned': 39})
heuristic: (10, 'DOWN', {'generated': 96, 'expanded': 97, 'pruned': 47})


### full run

In [11]:
def print_result(name, move, value, stats, depth, t, show_pruned):
    print(f'Algorithm: {name}')
    print(f'Best Move: {move}')
    print(f'Evaluation Score = {value}')
    print(f'Nodes Generated = {stats["generated"]}')
    print(f'Nodes Expanded = {stats["expanded"]}')
    if show_pruned:
        print(f'Nodes Pruned = {stats["pruned"]}')
    print(f'Search Depth = {depth}')

In [12]:
t0 = time.time()
mm_v, mm_m, mm_s = search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, False, False)
print_result('Minimax', mm_m, mm_v, mm_s, 5, time.time()-t0, False)
print()
t0 = time.time()
ab_v, ab_m, ab_s = search(grid1, 5, 7, a1, b1, e1, 'MAX', 5, True, False)
print_result('Alpha-Beta', ab_m, ab_v, ab_s, 5, time.time()-t0, True)
print()
print('Comparison:')
print(f'Nodes expanded -> Minimax = {mm_s["expanded"]}, Alpha-Beta = {ab_s["expanded"]}')
print(f'Nodes pruned -> Alpha-Beta = {ab_s["pruned"]}')

Algorithm: Minimax
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 66
Nodes Expanded = 67
Search Depth = 5

Algorithm: Alpha-Beta
Best Move: RIGHT
Evaluation Score = 1
Nodes Generated = 35
Nodes Expanded = 36
Nodes Pruned = 9
Search Depth = 5

Comparison:
Nodes expanded -> Minimax = 67, Alpha-Beta = 36
Nodes pruned -> Alpha-Beta = 9
